# Frozen-trunk π0 training — Colab A100

Run the cells in order. **Runtime → Change runtime type → A100** first.

### How this survives disconnection
Training runs as a **background process**, not in a cell, so an interrupted or
restarted cell does not kill it. Cell 6 is only a viewer.

What you cannot survive is the **runtime being recycled** — Colab reclaims it on
idle, and at ~12 h (Pro) / ~24 h (Pro+). That is why checkpoints are mirrored to
Drive and cell 7 resumes from the last one.

### Colab-specific traps this notebook works around
* `/content` is **wiped** on every restart — the 14 GB model is re-pulled each
  session (fast on Colab's link). Only checkpoints go to Drive.
* **Drive free tier is 15 GB** and each π0 checkpoint is several GB, so only the
  newest is kept there. Everything else lives on `/content`.
* Writing checkpoints *directly* to Drive stalls training — Drive I/O is slow
  and blocking. We write to local disk and mirror.

In [ ]:
#@title 1. Check the GPU you actually got
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print('\nvCPUs:', subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
!df -h /content | tail -1
if 'A100' not in name:
    print(f'\nWARNING: this is a {name}, not an A100.')
    print('T4 (16GB) will NOT fit frozen-trunk pi0 at batch 16.')
    print('Runtime -> Change runtime type -> A100, or drop batch_size to 4-8.')

In [ ]:
#@title 2. Mount Drive (checkpoints only) + install pinned deps
from google.colab import drive
drive.mount('/content/drive')

import pathlib
CKPT_DRIVE = pathlib.Path('/content/drive/MyDrive/skygrip_ckpt')
CKPT_DRIVE.mkdir(parents=True, exist_ok=True)
print('checkpoint mirror:', CKPT_DRIVE)
!df -h /content/drive | tail -1

# Pinned to the versions the dataset was produced with (codebase_version v3.0).
!pip install -q "lerobot==0.6.0"
import torch, lerobot
print('torch', torch.__version__, '| lerobot', lerobot.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
#@title 3. Hugging Face auth
# Use your FINE-GRAINED token (no expiry). It needs:
#   - repo.write on your account  (to push the trained checkpoint)
#   - 'Read access to public gated repos'  <-- pi0 pulls gated PaliGemma;
#     without it training dies at model load with a 403 that looks like a
#     network error.
from huggingface_hub import notebook_login, whoami
notebook_login()

In [ ]:
#@title 4. Pull weights + data (fails fast, before any GPU time)
import os, json, pathlib
os.environ['HF_HOME'] = '/content/hf'          # ephemeral but fast; NOT Drive

from huggingface_hub import snapshot_download, hf_hub_download, whoami
w = whoami(); a = w.get('auth', {}).get('accessToken', {})
print(f"[auth] {w.get('name')} | {a.get('displayName')} | expiry {a.get('expiration') or 'none'}")
fg = a.get('fineGrained')
if fg and not fg.get('canReadGatedRepos'):
    print('[auth] WARNING: token cannot read gated repos -> pi0 will fail on PaliGemma')

print('\n[model] lerobot/pi0_base (~14 GB)')
snapshot_download('lerobot/pi0_base')

from lerobot.datasets.lerobot_dataset import LeRobotDataset
REPO = 'hanapasta/pick_hold_v4s_train'
ds = LeRobotDataset(REPO)
print(f'\n[data] {ds.num_episodes} episodes, {ds.num_frames} frames, fps {ds.fps}')
assert ds.num_episodes == 480, f'expected 480, got {ds.num_episodes}'

from lerobot.utils.feature_utils import dataset_to_policy_features
from lerobot.configs.types import FeatureType
feats = dataset_to_policy_features(ds.meta.features)
for k, v in feats.items():
    print(f'    {v.type.name:6s} {k:34s} {tuple(v.shape)}')
print('cameras', sum(1 for v in feats.values() if v.type == FeatureType.VISUAL))

p = hf_hub_download(REPO, 'episodes_meta.json', repo_type='dataset', local_dir='/content')
meta = json.loads(pathlib.Path(p).read_text())['episodes']
modes = {}
for e in meta: modes[e['mode']] = modes.get(e['mode'], 0) + 1
print('sidecar:', len(meta), 'records | modes', modes)

In [ ]:
#@title 5. Launch training in the BACKGROUND
import os, subprocess, pathlib, json

TAG        = 's1'      #@param {type:"string"}
BATCH_SIZE = 16        #@param {type:"integer"}
STEPS      = 30000     #@param {type:"integer"}
SAVE_FREQ  = 2000      #@param {type:"integer"}

OUT = f'/content/pickhold_pi0_frozen_{TAG}'
LOG = f'/content/train_{TAG}.log'
WORKERS = max(1, os.cpu_count() - 1)   # video-backed data: decoding is CPU work

rename = json.dumps({
    'observation.images.camera3': 'observation.images.base_0_rgb',
    'observation.images.camera1': 'observation.images.left_wrist_0_rgb',
    'observation.images.camera2': 'observation.images.right_wrist_0_rgb'})

# Written to a script rather than inlined, so the JSON quoting cannot be
# mangled by the shell.
script = f'''#!/usr/bin/env bash
export HF_HOME=/content/hf
lerobot-train \\
  --dataset.repo_id=hanapasta/pick_hold_v4s_train \\
  --policy.path=lerobot/pi0_base \\
  --rename_map='{rename}' \\
  --output_dir={OUT} \\
  --batch_size={BATCH_SIZE} \\
  --steps={STEPS} \\
  --save_freq={SAVE_FREQ} \\
  --num_workers={WORKERS} \\
  --policy.device=cuda \\
  --policy.dtype=bfloat16 \\
  --policy.train_expert_only=true \\
  --wandb.enable=false \\
  --policy.push_to_hub=false 2>&1 | tee -a {LOG}
'''
pathlib.Path('/content/train.sh').write_text(script)

if pathlib.Path(OUT).exists():
    print(f'{OUT} exists -- use cell 7 to RESUME, or change TAG')
else:
    subprocess.Popen(['setsid', 'nohup', 'bash', '/content/train.sh'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    print(f'launched in background\n  out {OUT}\n  log {LOG}')
    print(f'  batch {BATCH_SIZE}, workers {WORKERS}, frozen trunk (expert only)')
    print('\nNow run cell 6. If it OOMs, lower BATCH_SIZE or add')
    print('--policy.gradient_checkpointing=true to the script above.')

In [ ]:
#@title 6. Monitor + mirror checkpoints to Drive (leave running)
# This cell is ONLY a viewer plus a Drive mirror. Interrupting it does not
# touch training. It also keeps the session active.
import re, subprocess, time, shutil, pathlib
from IPython.display import clear_output

TAG = 's1'
OUT = pathlib.Path(f'/content/pickhold_pi0_frozen_{TAG}')
LOG = pathlib.Path(f'/content/train_{TAG}.log')
MIRROR = pathlib.Path('/content/drive/MyDrive/skygrip_ckpt') / TAG
MIRROR.mkdir(parents=True, exist_ok=True)
TOTAL = 30000
_step = re.compile(r'step[:=\s]+(\d[\d,]*)', re.I)

def mirror_latest():
    """Keep ONLY the newest checkpoint on Drive -- the free tier is 15 GB and
    each pi0 checkpoint is several GB."""
    cks = sorted((OUT / 'checkpoints').glob('[0-9]*')) if (OUT / 'checkpoints').exists() else []
    if not cks:
        return 'none yet'
    latest = cks[-1]
    dst = MIRROR / latest.name
    if not dst.exists():
        for old in MIRROR.glob('[0-9]*'):
            shutil.rmtree(old, ignore_errors=True)
        shutil.copytree(latest, dst)
        return f'mirrored {latest.name} -> Drive'
    return f'Drive has {dst.name}'

while True:
    tail = subprocess.run(['tail','-n','30',str(LOG)], capture_output=True,
                          text=True).stdout if LOG.exists() else '(no log yet)'
    g = subprocess.run(['nvidia-smi',
         '--query-gpu=utilization.gpu,memory.used,memory.total',
         '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
    util = int(g.split(',')[0]) if g else 0
    alive = bool(subprocess.run(['pgrep','-f','lerobot-train'],
                 capture_output=True, text=True).stdout.strip())
    hits = _step.findall(tail)
    prog = f'step {int(hits[-1].replace(",","")):,}/{TOTAL:,}' if hits else ''
    clear_output(wait=True)
    print(f'GPU {g}   {"<-- DECODE-BOUND? raise num_workers" if util < 70 else ""}')
    print(f'running: {alive}   {prog}')
    print(mirror_latest())
    print('-'*72)
    print(tail)
    if not alive and (OUT/'checkpoints').exists():
        print('\n*** training stopped -- use cell 7 to resume')
        break
    time.sleep(30)

In [ ]:
#@title 7. RESUME after the runtime was recycled
# Run cells 1-4 first (the runtime is fresh, so /content is empty), then this.
import pathlib, shutil, subprocess
TAG = 's1'
OUT = pathlib.Path(f'/content/pickhold_pi0_frozen_{TAG}')
MIRROR = pathlib.Path('/content/drive/MyDrive/skygrip_ckpt') / TAG

cks = sorted(MIRROR.glob('[0-9]*'))
assert cks, f'no checkpoint on Drive at {MIRROR}'
latest = cks[-1]
print('restoring', latest.name, 'from Drive')
(OUT / 'checkpoints').mkdir(parents=True, exist_ok=True)
if not (OUT / 'checkpoints' / latest.name).exists():
    shutil.copytree(latest, OUT / 'checkpoints' / latest.name)
last = OUT / 'checkpoints' / 'last'
if last.exists() or last.is_symlink():
    last.unlink()
last.symlink_to(OUT / 'checkpoints' / latest.name)

cfg = last / 'pretrained_model' / 'train_config.json'
assert cfg.exists(), f'missing {cfg}'
script = f'''#!/usr/bin/env bash
export HF_HOME=/content/hf
lerobot-train --config_path={cfg} --resume=true 2>&1 | tee -a /content/train_{TAG}.log
'''
pathlib.Path('/content/resume.sh').write_text(script)
subprocess.Popen(['setsid','nohup','bash','/content/resume.sh'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 start_new_session=True)
print('resumed from', latest.name, '-- run cell 6 to watch')

In [ ]:
#@title 8. Push the finished checkpoint to the Hub
# Colab storage is not a backup. Push so the interpretability chain can load it
# from anywhere (including back on the cluster).
import pathlib
TAG = 's1'
OUT = pathlib.Path(f'/content/pickhold_pi0_frozen_{TAG}')
ck = OUT/'checkpoints'/'last'/'pretrained_model'
if not ck.exists():
    ck = sorted((OUT/'checkpoints').glob('*/pretrained_model'))[-1]
print('loading', ck)

from lerobot.policies.pi0.modeling_pi0 import PI0Policy
policy = PI0Policy.from_pretrained(str(ck))
n_all = sum(p.numel() for p in policy.parameters())
n_tr  = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f'{n_all/1e9:.2f}B params, {n_tr/1e6:.0f}M trainable ({100*n_tr/n_all:.1f}%)'
      ' -- expect ~10% for a frozen trunk')

REPO = f'hanapasta/pickhold-pi0-frozen-{TAG}'
policy.push_to_hub(REPO, private=True)
print('pushed', REPO)
print('\nNext (see INTERPRETABILITY_README.md):')
print(f'  python probe_extract_pi0.py --checkpoint {REPO} \\')
print('      --dataset hanapasta/pick_hold_v4s_train \\')
print('      --meta episodes_meta.json --out decision_activations.npz')